### Hex-Max – Lösung Schritt für Schritt

*40. Bundeswettbewerb Informatik (2021/2022), 2. Runde, Aufgabe 3*

Severin hat heute in der Schule das Hexadezimalsystem kennengelernt. Er verwendet nun kurze Stäbchen, um die sechzehn Ziffern
darzustellen, wie in einer Siebensegmentanzeige. Für die Darstellung einer Ziffer stehen also sieben Positionen zur Verfügung;
jede Position ist entweder durch ein Stäbchen belegt oder sie ist frei.

Severin will nun das folgende, von der Lehrerin gestellte Rätsel lösen: Gegeben ist eine Hexadezimalzahl (kurz: Hex-Zahl) mit
n Ziffern. Gesucht ist die größte Hex-Zahl mit der gleichen Anzahl an Ziffern, die aus der gegebenen Zahl durch eine zusätzlich
gegebene Maximalzahl an Umlegungen erzeugt werden kann. „Umlegung“ bedeutet, ein Stäbchen von seiner bisherigen Position an eine
andere, bislang freie Position zu bewegen.

Eine einzelne Umlegung muss noch keine gültige Hex-Zahl ergeben, aber das Ergebnis nach allen Umlegungen muss eine gültige
Hex-Zahl in der Siebensegmentdarstellung sein. Zu keiner Zeit darf die Darstellung einer Ziffer komplett „geleert“ werden. Die
gegebene Maximalzahl an Umlegungen muss nicht ausgeschöpft werden.

Hier ein Beispiel: Gegeben sind die Zahl D24 und die Maximalzahl an Umlegungen 3. Die gesuchte Hex-Zahl ist EE4.

#### Aufgabe

Schreibe ein Programm, das eine Hex-Zahl sowie die Maximalzahl `m` an Umlegungen einliest und die größte Hexadezimalzahl ermittelt,
die mit höchstens `m` Umlegungen erzeugt werden kann. Das Programm soll nach jeder Umlegung den Zwischenstand, also die aktuelle
Belegung der Positionen ausgeben.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `hexmax0.txt` bis `hexmax5.txt`. Die erste Zeile enthält die Hex-Zahl, die zweite die Maximalzahl
`m` der Umlegungen (die Dateien haben keine Formatbeschreibung; so lesen wir sie).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zahlen vergleicht man von links:** Eine größere Ziffer vorn macht die Zahl größer, egal was dahinter steht. Darum probiert man
  die Ziffern **von links nach rechts** jeweils von `F` abwärts (Greedy). Dabei muss man nur prüfen: „Lässt sich der Rest noch zu
  gültigen Ziffern machen?“
- **Umformulieren:** Statt „Stäbchen umlegen“ zählt man nur, an wie vielen Positionen sich die Belegung ändert. Das macht aus dem
  Rätsel eine Rechenaufgabe (zwei Summen).
- **Dynamische Programmierung (Tabelle von hinten):** Eine Tabelle merkt sich für jeden Rest der Zahl, was dort mindestens nötig
  ist, damit man sie später nicht neu berechnet.
- **Mengen:** Eine Ziffer ist die Menge ihrer belegten Positionen. Mit `-` (Differenz) sieht man, was weg- und dazukommt.

**Nützliche Funktionen und Bibliotheken**
- `set` und die Differenz `a - b`, `len(...)` für die Zahl der Stäbchen.
- Listen mit `[INF] * k` füllen, Teillisten mit `liste[k:]` verschieben und `map(min, liste1, liste2)` für das elementweise Minimum
  (damit rechnet die Tabelle viel schneller als mit doppelten Schleifen).
- `itertools.product` für alle Hex-Zahlen einer kleinen Länge (zum Prüfen).

## Die Idee

**Die Ziffern.** Die sieben Positionen nennen wir `a` bis `g`: `a` oben, `b` rechts oben, `c` rechts unten, `d` unten, `e` links
unten, `f` links oben, `g` in der Mitte. Jede Hex-Ziffer ist eine Menge dieser Buchstaben. Das Bild der Aufgabe fehlt in unseren
Unterlagen; wir nehmen die übliche Anzeige (`B` und `D` als „b“ und „d“, `7` ohne Stäbchen `f`). Das Beispiel `D24 → EE4` und alle
Ergebnisse der Lösungshinweise kommen damit heraus, die Annahme passt also.

**Umformulieren.** Wie viele Umlegungen braucht eine Zielzahl `B`, wenn man von `A` ausgeht? Sei `x` die Zahl der Positionen, an
denen sich die Belegung ändert (Stäbchen weg oder dazu). Die Zahl der Stäbchen bleibt gleich, also sind genauso viele Stäbchen
**abgehoben** wie **abgelegt** worden: `x = 2 · l`, und jede Umlegung hebt eins ab und legt eins ab, also genügen `l` Umlegungen.
Umgekehrt kann man jede Folge von höchstens `m` Umlegungen so lesen. Wir suchen deshalb die größte Zahl `B` mit

1. gleich vielen Stäbchen wie `A`,
2. höchstens `2 · m` geänderten Positionen.

(Dass man die Stäbchen auch ohne leere Ziffer umlegen kann, zeigt der Abschnitt „Die einzelnen Umlegungen“.)

**Greedy mit Tabelle.** Wir gehen von links nach rechts und nehmen für jede Stelle die größte Ziffer, bei der die Zahl noch zu
retten ist. Dazu brauchen wir für jeden Rest der Zahl (Ziffern ab `i`) die Antwort auf: „Wie viele Positionen muss ich dort
mindestens ändern, wenn ich dort genau `y` Stäbchen übrig haben will (oder fehlen lasse, wenn `y` negativ ist)?“ Das ist die Tabelle
`e(i, y)`; sie wird von hinten nach vorn ausgefüllt.

## Schritt 1: Ziffern als Mengen, Einlesen

`r(a, b)` ist die Zahl der **übrigen** Stäbchen, wenn Ziffer `a` zu `b` wird (negativ: es fehlen welche), `s(a, b)` die Zahl der
geänderten Positionen. `D → E` zum Beispiel: `D = bcdeg`, `E = adefg`; weg `b`, `c`; dazu `a`, `f`. Also `r = 0` und `s = 4`.

In [ ]:
SEGMENTE = {'0': 'abcdef', '1': 'bc', '2': 'abdeg', '3': 'abcdg', '4': 'bcfg', '5': 'acdfg', '6': 'acdefg', '7': 'abc',
            '8': 'abcdefg', '9': 'abcdfg', 'A': 'abcefg', 'B': 'cdefg', 'C': 'adef', 'D': 'bcdeg', 'E': 'adefg', 'F': 'aefg'}
ZIFFERN = '0123456789ABCDEF'
ZIFFER = {z: set(SEGMENTE[z]) for z in ZIFFERN}


def uebrig(a, b):
    return len(ZIFFER[a] - ZIFFER[b]) - len(ZIFFER[b] - ZIFFER[a])


def geaendert(a, b):
    return len(ZIFFER[a] - ZIFFER[b]) + len(ZIFFER[b] - ZIFFER[a])


def lies_hexmax(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split()
    return zeilen[0], int(zeilen[1])


print(uebrig('D', 'E'), geaendert('D', 'E'))
print(uebrig('1', '8'), geaendert('1', '8'))       # die 1 hat 2 Stäbchen, die 8 hat 7: 5 fehlen

0 4
-5 5


## Schritt 2: Die Tabelle `e(i, y)`

`e[i][y]` = die kleinste Zahl geänderter Positionen in den Ziffern `A[i], A[i+1], …, A[n-1]` (Zählung ab 0), wenn diese Ziffern zu
gültigen Hex-Ziffern werden und dabei `y` Stäbchen übrig bleiben. Geht das nicht, steht `UNMOEGLICH` (eine sehr große Zahl) da.

- Hinter der letzten Ziffer: `e[n][0] = 0`, alles andere unmöglich.
- Für Ziffer `i` probieren wir alle 16 Ziffern `b`: `e[i][y] = min über b ( e[i+1][y − r(A[i], b)] + s(A[i], b) )`.

Es genügt, `y` von `−m` bis `m` zu betrachten: Wenn rechts `y` Stäbchen übrig bleiben, müssen sie links wieder abgelegt werden (und
umgekehrt). Das sind mindestens `y` Änderungen rechts und `y` links, zusammen `2y`, und mehr als `2m` Änderungen sind nicht erlaubt. Von allen Ziffern `b` mit demselben `r` zählt nur die mit dem
kleinsten `s`. Damit hat jede Ziffer nur wenige Möglichkeiten (`OPTIONEN`). Die Listen werden um `r` verschoben und elementweise
verglichen, das spart die innere Schleife in Python.

In [ ]:
UNMOEGLICH = 10 ** 9


def optionen(a):
    """Für jede Ziffer a: die Paare (r, s), bei gleichem r nur das kleinste s."""
    beste = {}
    for b in ZIFFERN:
        r, s = uebrig(a, b), geaendert(a, b)
        if r not in beste or s < beste[r]:
            beste[r] = s
    return sorted(beste.items())


OPTIONEN = {a: optionen(a) for a in ZIFFERN}
print(OPTIONEN['D'])

[(-2, 2), (-1, 3), (0, 0), (1, 3), (2, 4), (3, 3)]


`OPTIONEN['D']` sagt: Aus einem `D` kann man mit 0 Änderungen `r = 0` erreichen (es bleibt ein `D`), mit 2 Änderungen `r = −2` (zum
Beispiel `8`, es fehlen dann 2 Stäbchen), mit 3 Änderungen `r = 3` (zur `1` bleiben 3 Stäbchen übrig) usw. Andere Paare `(r, s)` mit
größerem `s` werden nicht gebraucht.

In [ ]:
def berechne_e(A, m):
    n = len(A)
    breite = 2 * m + 1                     # Index j = y + m, y von -m bis m
    e = [None] * (n + 1)
    e[n] = [UNMOEGLICH] * breite
    e[n][m] = 0
    for i in range(n - 1, -1, -1):
        alt = e[i + 1]
        neu = [UNMOEGLICH] * breite
        for r, s in OPTIONEN[A[i]]:
            if r >= 0:                     # neues y = altes y + r
                kandidat = [UNMOEGLICH] * min(r, breite) + [v + s for v in alt[:breite - r]]
            else:
                kandidat = [v + s for v in alt[-r:]] + [UNMOEGLICH] * min(-r, breite)
            neu = list(map(min, neu, kandidat))
        e[i] = neu
    return e


def min_aenderungen(e, i, y, m):
    """e(i, y) mit Prüfung, dass y im Bereich -m..m liegt."""
    if -m <= y <= m:
        return e[i][y + m]
    return UNMOEGLICH


A0, M0 = lies_hexmax('beispieldaten/hexmax0.txt')
e0 = berechne_e(A0, M0)
print(A0, M0)
for i in range(len(A0) + 1):
    print('ab Stelle', i, ':', {y: min_aenderungen(e0, i, y, M0) for y in range(-M0, M0 + 1) if min_aenderungen(e0, i, y, M0) < UNMOEGLICH})

D24 3
ab Stelle 0 : {-3: 3, -2: 2, -1: 3, 0: 0, 1: 3, 2: 2, 3: 3}
ab Stelle 1 : {-3: 3, -2: 2, -1: 3, 0: 0, 1: 3, 2: 2, 3: 5}
ab Stelle 2 : {-3: 3, -2: 2, -1: 3, 0: 0, 1: 3, 2: 2}
ab Stelle 3 : {0: 0}


Lesehilfe für `D24` mit `m = 3`: Ab Stelle 3 (hinter der letzten Ziffer) geht nur `y = 0` mit 0 Änderungen. Ab Stelle 2 (nur die `4`) geht
`y = 0` mit 0 Änderungen (die `4` bleibt) und `y = 2` mit 2 Änderungen (aus `4 = bcfg` wird `1 = bc`: `f` und `g` fallen weg). Ab Stelle 0
(ganze Zahl) sind es dieselben Werte wie bei Stelle 1 und Stelle 2, nur dass auch `y = 3` mit 3 Änderungen geht.

## Schritt 3: Greedy von links

Wir wählen Ziffer für Ziffer die größte Ziffer `b` (von `F` abwärts), mit der noch eine Lösung möglich ist. Wir merken uns

- `x` = bisher geänderte Positionen und
- `y` = bisher übrig gebliebene Stäbchen (ist `y < 0`, haben wir Stäbchen von rechts geliehen; ist `y > 0`, müssen die Ziffern rechts noch
  `y` Stäbchen aufnehmen).

`b` ist erlaubt, wenn `x + s(A[i], b) + e(i+1, −(y + r(A[i], b))) ≤ 2m`: Die Ziffern rechts müssen insgesamt `−y` Stäbchen übrig
behalten (`e` ist für diesen Wert abzufragen), und mit den bisherigen Änderungen insgesamt nicht mehr als `2m` kosten. Die Schleife findet immer eine Ziffer, denn die bisherige Wahl war so
getroffen, dass die Tabelle eine Fortsetzung kennt; im schlimmsten Fall ist das die ursprüngliche Ziffer `A[i]`.

In [ ]:
def groesste_zahl(A, m):
    e = berechne_e(A, m)
    n = len(A)
    ergebnis = ''
    x = 0                                  # bisher geänderte Positionen
    y = 0                                  # bisher übrige Stäbchen
    for i in range(n):
        for b in reversed(ZIFFERN):        # von F abwärts
            x2 = x + geaendert(A[i], b)
            y2 = y + uebrig(A[i], b)
            if x2 + min_aenderungen(e, i + 1, -y2, m) <= 2 * m:
                ergebnis += b
                x, y = x2, y2
                break
    return ergebnis, x // 2                # x ist gerade, x // 2 Umlegungen


print(groesste_zahl('D24', 3))

('EE4', 3)


Das ist die Zahl `EE4` der Aufgabe; sie braucht 3 Umlegungen. Zur Kontrolle probieren wir für kleine Zahlen **alle** Hex-Zahlen durch
(`itertools.product`): Eine Zahl `B` ist erreichbar, wenn sie gleich viele Stäbchen hat und `x ≤ 2m` ist. Das ist von der Umformulierung
oben abhängig, dient also nur dazu, Fehler in der Tabelle zu finden.

In [ ]:
from itertools import product
import random


def groesste_zahl_langsam(A, m):
    beste = None
    for B in product(ZIFFERN, repeat=len(A)):
        stab_a = sum(len(ZIFFER[a]) for a in A)
        stab_b = sum(len(ZIFFER[b]) for b in B)
        x = sum(geaendert(a, b) for a, b in zip(A, B))
        if stab_a == stab_b and x <= 2 * m:
            if beste is None or B > beste:
                beste = B
    return ''.join(beste)


random.seed(1)
anzahl = 0
for _ in range(60):
    n = random.randint(1, 3)
    A = ''.join(random.choice(ZIFFERN) for _ in range(n))
    for m in range(0, 7):
        assert groesste_zahl(A, m)[0] == groesste_zahl_langsam(A, m), (A, m)
        anzahl += 1
print(anzahl, 'Vergleiche, alle gleich')

420 Vergleiche, alle gleich


## Schritt 4: Die einzelnen Umlegungen

Jetzt kennen wir die Zielzahl `B`. Die Aufgabe verlangt aber auch jede einzelne Umlegung und den Zwischenstand. Das geht so
(Beweisidee der Lösungshinweise):

1. **Innerhalb jeder Ziffer:** Sind bei Ziffer `i` Stäbchen abzuheben (`A[i] − B[i]`) **und** Positionen zu belegen (`B[i] − A[i]`), lege
   Stäbchen von der einen Seite auf die andere. So wird die Ziffer nie leer.
2. **Zwischen Ziffern:** Nun hat jede Ziffer nur noch abzuhebende **oder** nur noch zu belegende Positionen. Wir sammeln alle
   abzuhebenden (Ziffer, Position) in einer Liste, alle zu belegenden in einer zweiten, und legen die Stäbchen paarweise um.
   Eine Ziffer, von der nur abgehoben wird, wird am Ende die nichtleere Ziffer `B[i]`, also nie leer.

Es sind genau `x / 2` Umlegungen, nicht mehr. `spiele_nach` prüft eine Folge: Das Stäbchen liegt wirklich an der Startposition, das Ziel
ist frei, keine Ziffer wird leer und am Ende steht `B` da.

In [ ]:
def umlegungen(A, B):
    """Liste von Umlegungen (Ziffernnummer ab 0, von Position, nach Position)."""
    folge = []
    wegnehmen = []                         # (Ziffer, Position), die nach dem Schritt 1 noch abzuheben sind
    belegen = []                           # (Ziffer, Position), die noch zu belegen sind
    for i in range(len(A)):
        weg = sorted(ZIFFER[A[i]] - ZIFFER[B[i]])
        dazu = sorted(ZIFFER[B[i]] - ZIFFER[A[i]])
        while weg and dazu:                # Schritt 1: innerhalb der Ziffer
            folge.append((i, weg.pop(0), dazu.pop(0)))
        wegnehmen += [(i, p) for p in weg]
        belegen += [(i, p) for p in dazu]
    for (i1, p1), (i2, p2) in zip(wegnehmen, belegen):      # Schritt 2: Paare zwischen Ziffern
        folge.append((i1, p1, (i2, p2)))
    return folge


def spiele_nach(A, B, folge, ausgeben=False):
    """Führt die Umlegungen aus, prüft sie und gibt auf Wunsch jeden Zwischenstand aus."""
    stand = [set(ZIFFER[z]) for z in A]
    if ausgeben:
        print('Start:      ', ' '.join(''.join(sorted(s)) for s in stand))
    for nr, (i1, p1, ziel) in enumerate(folge, 1):
        i2, p2 = (i1, ziel) if isinstance(ziel, str) else ziel
        assert p1 in stand[i1] and p2 not in stand[i2]
        stand[i1].remove(p1)
        stand[i2].add(p2)
        assert stand[i1], 'Ziffer geleert'
        if ausgeben:
            print('Umlegung %2d:' % nr, ' '.join(''.join(sorted(s)) for s in stand), '  (Ziffer %d: %s -> Ziffer %d: %s)' % (i1 + 1, p1, i2 + 1, p2))
    assert stand == [ZIFFER[z] for z in B]
    return len(folge)


B0, n0 = groesste_zahl(A0, M0)
spiele_nach(A0, B0, umlegungen(A0, B0), ausgeben=True)

Start:       bcdeg abdeg bcfg
Umlegung  1: acdeg abdeg bcfg   (Ziffer 1: b -> Ziffer 1: a)
Umlegung  2: adefg abdeg bcfg   (Ziffer 1: c -> Ziffer 1: f)
Umlegung  3: adefg adefg bcfg   (Ziffer 2: b -> Ziffer 2: f)


Jede Zeile zeigt die belegten Positionen der drei Ziffern nach der Umlegung (`bcdeg` ist ein `D`, `adefg` ein `E`). Am Ende steht
`adefg adefg bcfg`, also `EE4`. Bei `D24` passiert alles innerhalb der Ziffern (Schritt 1): `b` nach `a`, `c` nach `f` in der ersten Ziffer und
`b` nach `f` in der zweiten. Wie Schritt 2 zwischen den Ziffern aussieht, zeigt das Beispiel `hexmax1.txt` weiter unten.

## Schritt 5: Alle Beispiele

Wir rechnen alle sechs Beispiele und prüfen jeweils die Umlegungsfolge mit `spiele_nach`. Für kurze Zahlen steht das ganze Ergebnis da,
bei den langen kürzen wir: Die Ergebnisse beginnen mit vielen `F` und enden mit einigen `8` (die `8` hat die meisten Stäbchen; die `F`
brauchen wenig, also bleiben am Ende Stäbchen übrig). Gezeigt wird die Zahl der `F` vorn, das Mittelstück und die Zahl der `8` hinten.

In [ ]:
ergebnisse = {}
for nr in range(6):
    A, m = lies_hexmax('beispieldaten/hexmax%d.txt' % nr)
    B, umleg = groesste_zahl(A, m)
    anzahl = spiele_nach(A, B, umlegungen(A, B))
    assert anzahl == umleg <= m
    ergebnisse[nr] = (A, m, B, anzahl)
    vorn = len(B) - len(B.lstrip('F'))
    hinten = len(B) - len(B.rstrip('8'))
    print('hexmax%d.txt: n = %d, m = %d, benutzte Umlegungen: %d' % (nr, len(A), m, anzahl))
    if len(B) <= 40:
        print('  ', B)
    else:
        print('  %d mal F, Mittelstück %s, %d mal 8' % (vorn, B[vorn:len(B) - hinten], hinten))

hexmax0.txt: n = 3, m = 3, benutzte Umlegungen: 3
   EE4
hexmax1.txt: n = 10, m = 8, benutzte Umlegungen: 8
   FFFEA97B55
hexmax2.txt: n = 40, m = 37, benutzte Umlegungen: 37
   FFFFFFFFFFFFFFFFD9A9BEAEE8EDA8BDA989D9F8
hexmax3.txt: n = 100, m = 121, benutzte Umlegungen: 121
  64 mal F, Mittelstück AA98BB8B9DFAFEAE888DD888AD8BA8EA, 4 mal 8
hexmax4.txt: n = 100, m = 87, benutzte Umlegungen: 87
  38 mal F, Mittelstück EB8DE88BAA8ADD888898E9BA88AD98988F898AB7AF7BDA8A61BA7D4AD8F, 3 mal 8
hexmax5.txt: n = 1000, m = 1369, benutzte Umlegungen: 1369
  671 mal F, Mittelstück 88EFA9EBE89EFA99FBDAA8E8EAD88AB899F8E8F9AA9E9AD88988EDA9A99888EDAD989A8BAFD8A, 252 mal 8


Die Ergebnisse stimmen mit denen der Lösungshinweise überein (Zeichen für Zeichen verglichen am Ende des Generators): `EE4`,
`FFFEA97B55` und für `hexmax2.txt` 16 mal `F`, dann `D9A9BEAEE8EDA8BDA989D9F8`. Bei `hexmax3.txt` bis `hexmax5.txt` zeigen die Hinweise
ebenfalls erst viele `F`, dann ein Mittelstück und am Ende `8`; die Zahlen decken sich.

In allen sechs Beispielen werden **alle** erlaubten `m` Umlegungen gebraucht (die Zahl der benutzten Umlegungen steht oben).

Dass die Bedingung „keine Ziffer wird geleert“ nie stört, zeigt `spiele_nach`: Es hat für alle sechs Beispiele die Umlegungen nachgespielt.

**Die Zwischenstände für `hexmax1.txt`** (die Hinweise zeigen sie für die Beispiele 0 bis 2; bei `hexmax2.txt` sind es 37 Zeilen,
das drucken wir nicht aus). Die Umlegungen 1 bis 3 geschehen innerhalb einer Ziffer, ab Umlegung 4 wandern Stäbchen zwischen Ziffern
(Ziffer 1 bis 3 geben ab, Ziffer 4 bis 7 nehmen auf):

In [ ]:
A1, m1, B1, anzahl1 = ergebnisse[1]
print(A1, '->', B1, '(m = %d)' % m1)
spiele_nach(A1, B1, umlegungen(A1, B1), ausgeben=True)

509C431B55 -> FFFEA97B55 (m = 8)
Start:       acdfg abcdef abcdfg adef bcfg abcdg bc cdefg acdfg acdfg
Umlegung  1: adefg abcdef abcdfg adef bcfg abcdg bc cdefg acdfg acdfg   (Ziffer 1: c -> Ziffer 1: e)
Umlegung  2: adefg acdefg abcdfg adef bcfg abcdg bc cdefg acdfg acdfg   (Ziffer 2: b -> Ziffer 2: g)
Umlegung  3: adefg acdefg acdefg adef bcfg abcdg bc cdefg acdfg acdfg   (Ziffer 3: b -> Ziffer 3: e)
Umlegung  4: aefg acdefg acdefg adefg bcfg abcdg bc cdefg acdfg acdfg   (Ziffer 1: d -> Ziffer 4: g)
Umlegung  5: aefg adefg acdefg adefg abcfg abcdg bc cdefg acdfg acdfg   (Ziffer 2: c -> Ziffer 5: a)
Umlegung  6: aefg aefg acdefg adefg abcefg abcdg bc cdefg acdfg acdfg   (Ziffer 2: d -> Ziffer 5: e)
Umlegung  7: aefg aefg adefg adefg abcefg abcdfg bc cdefg acdfg acdfg   (Ziffer 3: c -> Ziffer 6: f)
Umlegung  8: aefg aefg aefg adefg abcefg abcdfg abc cdefg acdfg acdfg   (Ziffer 3: d -> Ziffer 7: a)


## Zum Schluss: Mit MiniZinc

Die Umformulierung von oben passt direkt in ein Constraint-Modell. Du kennst von den Arbeitsblättern zu MiniZinc 2D-Arrays,
`sum … where`, `forall` und Variablen als Index. Hier ist neu, dass wir **nicht optimieren**, sondern die Suche so einstellen, dass die
**erste** gefundene Lösung schon die größte ist:

- `d[i]` ist die Ziffer an Stelle `i` (0 bis 15). Die Tabellen `S` und `R` enthalten `s(a, b)` und `r(a, b)` aus Schritt 1.
  `S[A[i], d[i]]` ist ein Zugriff mit einer **Variablen als Index**.
- Die beiden Bedingungen sind genau die Umformulierung: höchstens `2m` geänderte Positionen, Stäbchensumme gleich.
- `int_search(d, input_order, indomain_max, complete)` legt fest: Stellen von links nach rechts, jede mit dem **größten** möglichen Wert
  zuerst, vollständig mit Zurückgehen. Der Solver findet dann zuerst die lexikografisch größte Lösung, und das ist bei gleicher
  Länge die größte Zahl. Eine Zielfunktion `maximize sum(d[i] * 16^(n-i))` ginge auch, die Zahlen werden aber riesig.

```minizinc
int: n;
int: m;
array[1..n] of 0..15: A;                  % Ziffern der Eingabe (0..15)
array[0..15, 0..15] of int: S;            % S[a, b]: geänderte Positionen von a nach b
array[0..15, 0..15] of int: R;            % R[a, b]: übrige Stäbchen von a nach b
array[1..n] of var 0..15: d;              % die gesuchte Zahl

constraint sum(i in 1..n)(S[A[i], d[i]]) <= 2 * m;
constraint sum(i in 1..n)(R[A[i], d[i]]) = 0;

solve :: int_search(d, input_order, indomain_max, complete) satisfy;
output ["\(d)\n"];
```

Die Datei mit den Daten erzeugt die Python-Zelle `als_dzn`.

In [ ]:
def als_dzn(A, m):
    zahlen = [ZIFFERN.index(z) for z in A]
    s = [geaendert(a, b) for a in ZIFFERN for b in ZIFFERN]
    r = [uebrig(a, b) for a in ZIFFERN for b in ZIFFERN]
    return 'n = %d;\nm = %d;\nA = %s;\nS = array2d(0..15, 0..15, %s);\nR = array2d(0..15, 0..15, %s);' % (len(A), m, zahlen, s, r)


print(als_dzn('D24', 3)[:90], '...')

n = 3;
m = 3;
A = [13, 2, 4];
S = array2d(0..15, 0..15, [0, 4, 3, 3, 4, 3, 2, 3, 1, 2, 2,  ...


Ergebnis beim Erzeugen dieses Notebooks (Zeitlimit 20 bzw. 30 Sekunden pro Aufruf; Gecode und Chuffed, in MiniZinc über
*Solver configuration* gewählt). „gleich“ heißt: dieselbe Zahl wie die Python-Lösung, „?“: Zeitlimit, keine Antwort. Die Anfänge von
`hexmax2.txt` sind eigene Proben (die ersten `k` Ziffern, `m` im gleichen Verhältnis).

| Beispiel | Ziffern | m | Gecode | Chuffed |
|----------|---------|---|--------|---------|
| `hexmax0.txt` | 3 | 3 | gleich | gleich |
| `hexmax1.txt` | 10 | 8 | gleich | gleich |
| Anfang von hexmax2.txt, 8 Ziffern | 8 | 7 | gleich | gleich |
| Anfang von hexmax2.txt, 16 Ziffern | 16 | 15 | gleich | gleich |
| Anfang von hexmax2.txt, 24 Ziffern | 24 | 22 | gleich | gleich |
| Anfang von hexmax2.txt, 28 Ziffern | 28 | 26 | gleich | gleich |
| Anfang von hexmax2.txt, 32 Ziffern | 32 | 30 | ? | ? |
| `hexmax2.txt` | 40 | 37 | ? | ? |
| `hexmax3.txt` | 100 | 121 | ? | ? |
| `hexmax4.txt` | 100 | 87 | ? | ? |
| `hexmax5.txt` | 1000 | 1369 | ? | ? |

Das Modell ist richtig, aber nur für kurze Zahlen brauchbar: Bis etwa 28 Ziffern sind die Solver schnell fertig, ab etwa 32 Ziffern
und bei den echten Beispielen `hexmax2.txt` bis `hexmax5.txt` kommt nichts mehr heraus. Der Grund: Der Solver wählt die Ziffern von links
nach rechts und merkt erst weit hinten, dass der Rest keine passende Zahl mehr ergibt (zu wenige oder zu viele Stäbchen). Dann muss er
viele Stellen zurück. Unsere Tabelle `e` löst genau dieses Problem, denn sie weiß **vorher**, ob der Rest noch geht. Das ist ein
schönes Beispiel dafür, dass ein passender Algorithmus (hier dynamische Programmierung) einem allgemeinen Löser überlegen sein kann.

## Für Neugierige

- **Laufzeit:** Die Tabelle hat `n · (2m + 1)` Einträge, jeder braucht höchstens 16 Vergleiche; das Greedy-Verfahren selbst braucht nur
  `16 · n` Schritte. Insgesamt `O(n · m)`. Ersetzt man `m` durch `min(m, 3,5 · n)` (mehr als jedes Stäbchen einmal umlegen lohnt nie),
  wird es zu `O(n · min(n, m))`.
- **Erster Ansatz der Hinweise:** `dp(i, x, y)` ist die größte Hex-Zahl der ersten `i` Stellen mit genau `x` geänderten Positionen
  und `y` Stäbchen mehr. Sie speichert ganze Zahlen und braucht `O(n² · m²)`; für `hexmax5.txt` ist das viel zu langsam.
  Unsere Tabelle `e` (zweiter Ansatz der Hinweise) vergleicht nie ganze Zahlen.
- **Warum Greedy richtig ist:** Eine größere Ziffer vorn schlägt alles, was hinten passiert. Und `e` sagt genau, ob hinten noch eine
  gültige Zahl möglich ist. Deshalb ist die Ziffer, die wir wählen, wirklich die größte mögliche.